# Wave-temporal Sleipner runner\n\nThis notebook is for remote execution on Google Colab. It keeps the main research notebook sequence untouched and runs the new wave-consistent temporal branch from a Drive-backed workspace.

In [ ]:
from google.colab import drive\nfrom pathlib import Path\nimport os\nimport sys\n\ndrive.mount('/content/drive')\n\nWORKSPACE = Path('/content/drive/MyDrive/Propagation')\nif not WORKSPACE.exists():\n    raise FileNotFoundError(f'Workspace not found: {WORKSPACE}')\n\nos.chdir(WORKSPACE)\nprint('Workspace:', WORKSPACE)\nprint('Python:', sys.version)\nif sys.version_info < (3, 11):\n    raise RuntimeError('This repo currently requires Python 3.11 or newer.')

In [ ]:
required_paths = [\n    Path('examples/sleipner_manifest.p07_11inline.json'),\n    Path('examples/sleipner_manifest.p10_11inline.json'),\n    Path('examples/exports/raw/94p07mid.sgy'),\n    Path('examples/exports/raw/01p07mid.sgy'),\n    Path('examples/exports/raw/04p07mid.sgy'),\n    Path('examples/exports/raw/06p07mid.sgy'),\n    Path('examples/exports/raw/94p10mid.sgy'),\n    Path('examples/exports/raw/10p10mid.sgy')\n]\nmissing = [str(path) for path in required_paths if not path.exists()]\nif missing:\n    raise FileNotFoundError('Missing required benchmark files:\n' + '\n'.join(missing))\nprint('Required benchmark files found.')

In [ ]:
!python -m pip install -U pip setuptools wheel\n!python -m pip install -e .[viz]

In [ ]:
import torch\nprint('CUDA available:', torch.cuda.is_available())\nif torch.cuda.is_available():\n    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
!PYTHONPATH=src python -m ccs_monitoring.cli run-all --config configs/paper_wave_temporal_colab.yaml

In [ ]:
!PYTHONPATH=src python -m ccs_monitoring.cli evaluate-field --config configs/sleipner_p10_wave_temporal_colab.yaml

In [ ]:
!PYTHONPATH=src python -m ccs_monitoring.cli evaluate-field --config configs/sleipner_p07_wave_temporal_colab.yaml

In [ ]:
import json\nfrom pathlib import Path\n\ndef load_json(path):\n    path = Path(path)\n    if not path.exists():\n        print('Missing:', path)\n        return None\n    return json.loads(path.read_text())\n\nsynthetic = load_json('runs/paper_wave_temporal_colab/results/metrics.json')\np10 = load_json('runs/sleipner_p10_wave_temporal_colab/results/summary.json')\np07 = load_json('runs/sleipner_p07_wave_temporal_colab/results/summary.json')\n\nprint('Synthetic metrics path:', Path('runs/paper_wave_temporal_colab/results/metrics.json'))\nprint('p10 summary path:', Path('runs/sleipner_p10_wave_temporal_colab/results/summary.json'))\nprint('p07 summary path:', Path('runs/sleipner_p07_wave_temporal_colab/results/summary.json'))\n\nif synthetic is not None:\n    test_metrics = synthetic['test']['wave_temporal_ml']\n    ood_metrics = synthetic['ood']['wave_temporal_ml']\n    print('Synthetic test wave IoU:', test_metrics.get('iou'))\n    print('Synthetic OOD wave IoU:', ood_metrics.get('iou'))\n    print('Synthetic held-out test metrics:', synthetic['test'].get('wave_temporal_ml_held_out_prediction', {}))\n\nif p10 is not None:\n    print('p10 wave constrained summary:', p10.get('wave_temporal_constrained_summary', {}))\n\nif p07 is not None:\n    print('p07 sequence summary:', p07.get('sequence_method_summary', {}).get('wave_temporal_constrained', {}))\n    print('p07 held-out summary:', p07.get('wave_leave_one_out_summary', {}))